# Building Vector DB

Run this file to build Vector DB. 

# Before running 
In terminal, run the following code: 
```bash
python3 -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
ollama serve
ollama pull llama3.1:8b
```

In [1]:
# LOCAL CHANGE: No Google Colab API key setup needed
# We are using Ollama locally, so no OPENAI_API_KEY is required.

import os
from dotenv import load_dotenv

load_dotenv()

False

In [2]:
from pathlib import Path
from typing import List
import json
import re

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_ollama import ChatOllama

# Load documents as PDFs
from langchain_community.document_loaders import PyPDFLoader

# Chunk documents
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embedding Model
from langchain_community.embeddings import HuggingFaceEmbeddings

# Create vector DB
from langchain_community.vectorstores import FAISS

print("Imports loaded.")


/Users/julie/Documents/github/ds593_finalproject/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports loaded.


# Upload relevant files
Skip if already done. 

In [3]:
pdf_folder = Path("pdfs")
pdf_folder.mkdir(exist_ok=True)

pdf_files = sorted(pdf_folder.glob("*.pdf"))

print(f"PDF folder: {pdf_folder.resolve()}")
print(f"PDFs found: {len(pdf_files)}")

for pdf in pdf_files:
    print("-", pdf.name)

if not pdf_files:
    raise FileNotFoundError(
        "No PDFs found. Put your PDF files inside the local 'pdfs/' folder, then rerun this cell."
    )


PDF folder: /Users/julie/Documents/github/ds593_finalproject/pdfs
PDFs found: 45
- 3511265.3550437.pdf
- 3511265.3550438.pdf
- 3511265.3550439.pdf
- 3511265.3550440.pdf
- 3511265.3550441.pdf
- 3511265.3550442.pdf
- 3511265.3550443.pdf
- 3511265.3550444.pdf
- 3511265.3550445.pdf
- 3511265.3550446.pdf
- 3511265.3550447.pdf
- 3511265.3550448.pdf
- 3511265.3550449.pdf
- 3511265.3550450.pdf
- 3511265.3550451.pdf
- 3511265.3550452.pdf
- 3511265.3550497.pdf
- 3614407.3643696.pdf
- 3614407.3643697.pdf
- 3614407.3643698.pdf
- 3614407.3643699.pdf
- 3614407.3643700.pdf
- 3614407.3643701.pdf
- 3614407.3643702.pdf
- 3614407.3643703.pdf
- 3614407.3643704.pdf
- 3614407.3643705.pdf
- 3614407.3643706.pdf
- 3614407.3643707.pdf
- 3614407.3643708.pdf
- 3709025.3712206.pdf
- 3709025.3712207.pdf
- 3709025.3712208.pdf
- 3709025.3712209.pdf
- 3709025.3712210.pdf
- 3709025.3712211.pdf
- 3709025.3712212.pdf
- 3709025.3712213.pdf
- 3709025.3712214.pdf
- 3709025.3712215.pdf
- 3709025.3712216.pdf
- 3709025.3712217

In [4]:
# Agentic IEEE citation extraction from PDF tex

citation_llm = ChatOllama(
    model="llama3.2:3b",
    temperature=0
)

def safe_json_parse(text: str) -> dict:
    """
    Ollama sometimes returns extra text around JSON.
    This safely extracts the JSON object.
    """
    text = text.strip()

    text = re.sub(r"^```json", "", text)
    text = re.sub(r"^```", "", text)
    text = re.sub(r"```$", "", text).strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match:
            return json.loads(match.group())

    raise ValueError(f"No valid JSON found in LLM response:\n{text}")


def extract_pdf_front_matter(pdf_path: str, max_pages: int = 1) -> str:
    """
    Reads the first few pages of a PDF.
    Citation information is usually located there.
    """
    loader = PyPDFLoader(pdf_path)
    pages = loader.load()

    front_pages = pages[:max_pages]

    return "\n\n".join(
        f"--- Page {i + 1} ---\n{page.page_content}"
        for i, page in enumerate(front_pages)
    )


def citation_agent(pdf_path: str) -> dict:
    """
    Agent that reads the PDF text and generates a full IEEE citation.
    """
    front_matter = extract_pdf_front_matter(pdf_path)
    front_matter = front_matter[:6000]

    prompt = f"""
You are a citation extraction agent.

Your task is to read the PDF text and create a full IEEE-style citation.

Return ONLY valid JSON in this exact format:

{{
  "title": "",
  "authors": "",
  "venue": "",
  "year": "",
  "publisher": "",
  "doi": "",
  "ieee_citation": ""
}}

Rules:
- Use only information visible in the PDF text.
- Do not invent missing fields.
- If a field is missing, write "Unknown".
- The ieee_citation must be a complete IEEE-style reference.
- Use this general IEEE format:
  Authors, "Title," Venue, Publisher, Year, doi: DOI.
- If there is no DOI, omit the DOI from the ieee_citation.
- Return JSON only.

PDF text:
{front_matter}
"""

    response = citation_llm.invoke(prompt)

    if hasattr(response, "content"):
        raw = response.content
    else:
        raw = str(response)

    return safe_json_parse(raw)


print("Citation agent ready.")

Citation agent ready.


# Load documents as PDFs
Skip if already done

In [5]:
# Load PDFs and attach agent-generated IEEE citations

docs = []
pdf_citations = {}

for pdf_path in pdf_files:
    print(f"\nExtracting citation for: {pdf_path.name}")

    try:
        citation_data = citation_agent(str(pdf_path))
    except Exception as e:
        print(f"Could not extract citation for {pdf_path.name}: {e}")

        citation_data = {
            "title": pdf_path.stem,
            "authors": "Unknown",
            "venue": "Unknown",
            "year": "Unknown",
            "publisher": "Unknown",
            "doi": "Unknown",
            "ieee_citation": f'Unknown, "{pdf_path.stem}," Unknown, Unknown.'
        }

    pdf_citations[pdf_path.name] = citation_data

    loader = PyPDFLoader(str(pdf_path))
    pages = loader.load()

    for page in pages:
        page.metadata["source_file"] = pdf_path.name
        page.metadata["source_path"] = str(pdf_path)
        page.metadata["page"] = page.metadata.get("page", 0) + 1

        # Agent-generated citation metadata
        page.metadata["title"] = citation_data.get("title", "Unknown")
        page.metadata["authors"] = citation_data.get("authors", "Unknown")
        page.metadata["venue"] = citation_data.get("venue", "Unknown")
        page.metadata["year"] = citation_data.get("year", "Unknown")
        page.metadata["publisher"] = citation_data.get("publisher", "Unknown")
        page.metadata["doi"] = citation_data.get("doi", "Unknown")
        page.metadata["ieee_citation"] = citation_data.get("ieee_citation", "Unknown citation")

    docs.extend(pages)

print("\nTotal loaded pages:", len(docs))
print("Total PDFs:", len(pdf_files))

print("\nExtracted IEEE citations:")
for file_name, citation in pdf_citations.items():
    print(f"\n{file_name}")
    print(citation["ieee_citation"])


Extracting citation for: 3511265.3550437.pdf
Could not extract citation for 3511265.3550437.pdf: llama runner process has terminated: %!w(<nil>) (status code: 500)

Extracting citation for: 3511265.3550438.pdf
Could not extract citation for 3511265.3550438.pdf: llama runner process has terminated: %!w(<nil>) (status code: 500)

Extracting citation for: 3511265.3550439.pdf
Could not extract citation for 3511265.3550439.pdf: llama runner process has terminated: %!w(<nil>) (status code: 500)

Extracting citation for: 3511265.3550440.pdf


KeyboardInterrupt: 

# Chunk documents

In [ ]:
# Chunk documents using fixed size chunking
# from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

chunks = splitter.split_documents(docs)

print(f"Chunks created: {len(chunks)}")

# Embedding Model

In [ ]:
# Embedding Model. Using free tier AI
# from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

# Create Vector DB

In [ ]:
# Create vector DB
# from langchain_community.vectorstores import FAISS

vector_db = FAISS.from_documents(chunks, embeddings)

print("Vector DB created!")

In [ ]:
save_path = "fixed_rag_faiss_index"

vector_db.save_local(save_path)

print(f"Vector DB saved locally to: {Path(save_path).resolve()}")